### Creación dataset D202 (rCMB y pseudomáscaras)

In [ ]:
import os
import pandas as pd
import shutil
import json
from sklearn.model_selection import train_test_split

# --- CONFIGURACIÓN DE RUTAS ---
INPUT_CSV = "/media/PORT-DISK/Practicas/ADNI_Final_Training_Set.csv"

# Rutas de origen
REAL_IMAGES_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI/workdir_ADNI_subset/raw/positives"
REAL_LABELS_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI/workdir_ADNI_subset/pseudo_labels/positives" 

# Estructura nnU-Net v2
DATASET_ID = "202"
DATASET_NAME = f"Dataset{DATASET_ID}_RealCMB"
NNUNET_RAW = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI"
OUTPUT_DIR = os.path.join(NNUNET_RAW, DATASET_NAME)

RANDOM_SEED = 42

def organize_real_stratified():
    # 1. Crear carpetas necesarias
    for folder in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
        os.makedirs(os.path.join(OUTPUT_DIR, folder), exist_ok=True)

    # 2. Cargar datos y preparar estratificación
    df = pd.read_csv(INPUT_CSV)
    df_pos = df[df['Has_CMB'] == True].copy()

    # Creamos estratos basados en Fabricante y Bins de Edad
    df_pos['Age_Bin'] = pd.cut(df_pos['Age'], bins=[0, 70, 80, 120], labels=['<70', '70-80', '80+'])
    df_pos['Stratum'] = df_pos['Manufacturer'].astype(str) + "_" + df_pos['Age_Bin'].astype(str)

    # 3. Muestreo estratificado (80% / 20% -> 200 / 50)
    train_df, test_df = train_test_split(
        df_pos, 
        test_size=50, 
        random_state=RANDOM_SEED, 
        stratify=df_pos['Stratum']
    )

    def process_files(subset_df, split_type):
        """
        split_type: 'Tr' o 'Ts'
        """
        count = 0
        print(f"Procesando subconjunto {split_type} ({len(subset_df)} casos)...")
        
        for _, row in subset_df.iterrows():
            subj_id = str(row['LONI_IMG_ID_STR']).strip()
            
            # Rutas de origen (usando el ID original)
            src_img = os.path.join(REAL_IMAGES_DIR, f"{subj_id}.nii.gz")
            src_lbl = os.path.join(REAL_LABELS_DIR, f"{subj_id}.nii.gz") 
            
            # Rutas de destino (sin cambiar nombre, solo añadiendo _0000 a la imagen)
            dst_img = os.path.join(OUTPUT_DIR, f"images{split_type}", f"{subj_id}_0000.nii.gz")
            dst_lbl = os.path.join(OUTPUT_DIR, f"labels{split_type}", f"{subj_id}.nii.gz")

            if os.path.exists(src_img) and os.path.exists(src_lbl):
                # Usamos copyfile para evitar errores de permisos en disco externo
                shutil.copyfile(src_img, dst_img)
                shutil.copyfile(src_lbl, dst_lbl)
                count += 1
            else:
                if not os.path.exists(src_img): print(f"  ERROR: No existe imagen {src_img}")
                if not os.path.exists(src_lbl): print(f"  ERROR: No existe segmentación {src_lbl}")
        
        return count

    # 4. Ejecutar transferencias
    count_tr = process_files(train_df, "Tr")
    count_ts = process_files(test_df, "Ts")

    # 5. Generar dataset.json acorde a los resultados
    json_dict = {
        "channel_names": {"0": "T2*"},
        "labels": {"background": 0, "CMB": 1},
        "numTraining": count_tr,
        "file_ending": ".nii.gz",
        "name": DATASET_NAME,
        "reference": "ADNI Real CMB Stratified Selection",
        "description": "200 Training / 50 Test. Original filenames preserved."
    }
    
    with open(os.path.join(OUTPUT_DIR, "dataset.json"), 'w') as f:
        json.dump(json_dict, f, indent=4)

    print(f"\nProceso finalizado.")
    print(f"Entrenamiento (Tr): {count_tr} sujetos.")
    print(f"Test (Ts): {count_ts} sujetos.")

if __name__ == "__main__":
    organize_real_stratified()

Procesando subconjunto Tr (200 casos)...
Procesando subconjunto Ts (50 casos)...

Proceso finalizado.
Entrenamiento (Tr): 200 sujetos.
Test (Ts): 50 sujetos.
